In [1]:
#Mounting Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
#Extracting ZIP
import zipfile
zip_path = "/content/drive/My Drive/fraud detection.zip"
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("/content/dataset/")

In [23]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
# Step 1: Import required libraries

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, roc_auc_score, classification_report

from imblearn.over_sampling import SMOTE

In [4]:
# Step 2: Load fraud detection dataset

df = pd.read_csv('/content/dataset/fraud.csv')
df.head()

,transaction_amount,hour_of_day,is_weekend,num_items,customer_age,prev_transactions,distance_from_home,device_type,network_quality,is_first_transaction,store_type,velocity_score,is_fraud
0,161.363691,3.0,0.0,2.0,18.000000,2.0,26.539742,1.0,48.403937,0.0,0.0,3.718296,0
1,116.202851,1.0,1.0,4.0,26.285818,2.0,50.714402,NaN,76.144979,0.0,0.0,4.951272,0
2,1.000000,2.0,0.0,5.0,18.000000,NaN,9.467935,0.0,67.600316,0.0,0.0,4.556043,0
3,48.780618,2.0,0.0,3.0,44.471190,NaN,41.077068,0.0,94.825526,0.0,0.0,6.918437,0
4,NaN,3.0,0.0,4.0,38.733609,8.0,NaN,2.0,100.000000,0.0,1.0,5.535335,1


In [5]:
# Step 3: Understand dataset structure

print("Dataset Shape:", df.shape)

print("\nColumn Names:")
print(df.columns)

print("\nData Types:")
print(df.dtypes)

Dataset Shape: (7000, 13)

Column Names:
Index(['transaction_amount', 'hour_of_day', 'is_weekend', 'num_items',
       'customer_age', 'prev_transactions', 'distance_from_home',
       'device_type', 'network_quality', 'is_first_transaction', 'store_type',
       'velocity_score', 'is_fraud'],
      dtype='object')

Data Types:
transaction_amount      float64
hour_of_day             float64
is_weekend              float64
num_items               float64
customer_age            float64
prev_transactions       float64
distance_from_home      float64
device_type             float64
network_quality         float64
is_first_transaction    float64
store_type              float64
velocity_score          float64
is_fraud                  int64
dtype: object


In [6]:
# Step 4: Check dataset information

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7000 entries, 0 to 6999
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   transaction_amount    6440 non-null   float64
 1   hour_of_day           6650 non-null   float64
 2   is_weekend            6860 non-null   float64
 3   num_items             6790 non-null   float64
 4   customer_age          6160 non-null   float64
 5   prev_transactions     6510 non-null   float64
 6   distance_from_home    6300 non-null   float64
 7   device_type           6720 non-null   float64
 8   network_quality       6370 non-null   float64
 9   is_first_transaction  6790 non-null   float64
 10  store_type            6860 non-null   float64
 11  velocity_score        5950 non-null   float64
 12  is_fraud              7000 non-null   int64  
dtypes: float64(12), int64(1)
memory usage: 711.1 KB


In [7]:
# Step 5: Check missing values

missing_values = df.isnull().sum()
print(missing_values)

transaction_amount       560
hour_of_day              350
is_weekend               140
num_items                210
customer_age             840
prev_transactions        490
distance_from_home       700
device_type              280
network_quality          630
is_first_transaction     210
store_type               140
velocity_score          1050
is_fraud                   0
dtype: int64


In [9]:
# Step 6: Remove missing values using median filling

for col in df.columns:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].median(), inplace=True)

print("Remaining Missing Values:")
print(df.isnull().sum())

Remaining Missing Values:
transaction_amount      0
hour_of_day             0
is_weekend              0
num_items               0
customer_age            0
prev_transactions       0
distance_from_home      0
device_type             0
network_quality         0
is_first_transaction    0
store_type              0
velocity_score          0
is_fraud                0
dtype: int64


In [10]:
# Step 7: Check whether dataset is labeled or not
print(df["is_fraud"].value_counts())

is_fraud
0    6279
1     721
Name: count, dtype: int64


In [11]:
# Step 8: Separate input features and output labels
X = df.drop("is_fraud", axis=1)
y = df["is_fraud"]

print(X.shape)
print(y.shape)

(7000, 12)
(7000,)


In [12]:
# Step 9: Check class imbalance
print(y.value_counts())
print("\nFraud Percentage:")
print(y.value_counts(normalize=True)*100)

is_fraud
0    6279
1     721
Name: count, dtype: int64

Fraud Percentage:
is_fraud
0    89.7
1    10.3
Name: proportion, dtype: float64


In [13]:
# Step 10: Split data into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


print(X_train.shape)
print(X_test.shape)

(5600, 12)
(1400, 12)


In [14]:
# Step 11: Handle class imbalance using SMOTE
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

print("Before SMOTE:")
print(y_train.value_counts())

print("\nAfter SMOTE:")
print(y_train_smote.value_counts())

Before SMOTE:
is_fraud
0    5023
1     577
Name: count, dtype: int64

After SMOTE:
is_fraud
0    5023
1    5023
Name: count, dtype: int64


In [15]:
# Step 12: Feature extraction and scaling
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_smote)
X_test_scaled = scaler.transform(X_test)

In [16]:
# Step 13: Train Logistic Regression classification model
log_model = LogisticRegression(
    max_iter=1000
)

log_model.fit(
    X_train_scaled,
    y_train_smote
)

LogisticRegression(max_iter=1000)

In [17]:
# Step 14: Evaluate Logistic Regression using Precision Recall ROC-AUC
log_prediction = log_model.predict(X_test_scaled)
log_probability = log_model.predict_proba(X_test_scaled)[:,1]

print("Precision:",
      precision_score(y_test, log_prediction))
print("Recall:",
      recall_score(y_test, log_prediction))
print("ROC-AUC:",
      roc_auc_score(y_test, log_probability))

print(classification_report(y_test, log_prediction))

Precision: 0.1111111111111111
Recall: 0.4930555555555556
ROC-AUC: 0.523581254423213
              precision    recall  f1-score   support

           0       0.90      0.55      0.68      1256
           1       0.11      0.49      0.18       144

    accuracy                           0.54      1400
   macro avg       0.51      0.52      0.43      1400
weighted avg       0.82      0.54      0.63      1400



In [18]:
# Step 15: Train Random Forest classification model
rf_model = RandomForestClassifier(
    random_state=42
)
rf_model.fit(
    X_train_smote,
    y_train_smote
)

RandomForestClassifier(random_state=42)

In [19]:
# Step 16: Evaluate Random Forest using Precision Recall ROC-AUC
rf_prediction = rf_model.predict(X_test)
rf_probability = rf_model.predict_proba(X_test)[:,1]

print("Precision:",
      precision_score(y_test, rf_prediction))
print("Recall:",
      recall_score(y_test, rf_prediction))
print("ROC-AUC:",
      roc_auc_score(y_test, rf_probability))

print(classification_report(y_test, rf_prediction))

Precision: 0.5
Recall: 0.006944444444444444
ROC-AUC: 0.5342052592002832
              precision    recall  f1-score   support

           0       0.90      1.00      0.95      1256
           1       0.50      0.01      0.01       144

    accuracy                           0.90      1400
   macro avg       0.70      0.50      0.48      1400
weighted avg       0.86      0.90      0.85      1400



In [20]:
# Step 17: Hyperparameter tuning Random Forest
parameters = {
    "n_estimators":[100,200],
    "max_depth":[5,10,None]
}
grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    parameters,
    scoring="roc_auc",
    cv=3
)
grid.fit(
    X_train_smote,
    y_train_smote
)
print("Best Parameters:")
print(grid.best_params_)

Best Parameters:
{'max_depth': None, 'n_estimators': 200}


In [21]:
# Step 18: Test optimized Random Forest model
best_model = grid.best_estimator_
final_prediction = best_model.predict(X_test)
final_probability = best_model.predict_proba(X_test)[:,1]

print("Final Precision:",
      precision_score(y_test, final_prediction))
print("Final Recall:",
      recall_score(y_test, final_prediction))

print("Final ROC-AUC:",
      roc_auc_score(y_test, final_probability))

Final Precision: 1.0
Final Recall: 0.006944444444444444
Final ROC-AUC: 0.5284163791578202


In [24]:
# Step 19: Test model with sample transaction prediction
sample = X_test.iloc[0].values.reshape(1,-1)
prediction = best_model.predict(sample)

if prediction[0] == 1:
    print("Transaction is Fraud")
else:
    print("Transaction is Genuine")

Transaction is Genuine
